# 🧬 Python Inheritance & Polymorphism — The Complete Guide

> **Module:** Object-Oriented Programming (OOP) | **Notebook 3 of 5**

Inheritance and Polymorphism are the twin pillars of Object-Oriented Architecture. They enable developers to model hierarchical relationships, reuse and extend code without modification (Open/Closed Principle), and interact with diverse object types through uniform interfaces.

This notebook provides an in-depth, hands-on exploration of Python's inheritance model: from single and multilevel inheritance to the Diamond Problem, Method Resolution Order (C3 Linearization), cooperative `super()`, Mixins, Duck Typing, and modern `typing.Protocol` structural subtyping.

---

## 📋 Table of Contents
1. [Introduction to Inheritance & The "Is-A" Relationship](#1.-Introduction-to-Inheritance-&-The-"Is-A"-Relationship)
2. [Single Inheritance & Method Overriding](#2.-Single-Inheritance-&-Method-Overriding)
3. [Multilevel & Hierarchical Inheritance](#3.-Multilevel-&-Hierarchical-Inheritance)
4. [Multiple Inheritance & The Diamond Problem](#4.-Multiple-Inheritance-&-The-Diamond-Problem)
5. [Method Resolution Order (MRO) & C3 Linearization](#5.-Method-Resolution-Order-(MRO)-&-C3-Linearization)
6. [Cooperative Multiple Inheritance with `super()`](#6.-Cooperative-Multiple-Inheritance-with-super())
7. [The Mixin Architectural Pattern](#7.-The-Mixin-Architectural-Pattern)
8. [Polymorphism: Duck Typing, Abstract Classes, & Protocols](#8.-Polymorphism:-Duck-Typing,-Abstract-Classes,-&-Protocols)
9. [Composition vs. Inheritance ("Favor Composition")](#9.-Composition-vs.-Inheritance-("Favor-Composition"))
10. [Real-World Case Studies & Architectural Patterns](#10.-Real-World-Case-Studies-&-Architectural-Patterns)
    - 10.1 [Enterprise Multi-Channel Notifier Hierarchy](#10.1-Enterprise-Multi-Channel-Notifier-Hierarchy)
    - 10.2 [Custom Container with `collections.abc`](#10.2-Custom-Container-with-collections.abc)
    - 10.3 [RPG Game Entity Engine with Mixins & Components](#10.3-RPG-Game-Entity-Engine-with-Mixins-&-Components)
11. [Common Pitfalls & Anti-Patterns](#11.-Common-Pitfalls-&-Anti-Patterns)
12. [Hands-On Interactive Challenges](#12.-Hands-On-Interactive-Challenges)
13. [Quick Reference Card & Summary](#13.-Quick-Reference-Card-&-Summary)


---
## 1. Introduction to Inheritance & The "Is-A" Relationship

### 🏛️ Core Concepts
- **Inheritance** allows a class (the *Derived / Child / Subclass*) to inherit attributes and methods from another class (the *Base / Parent / Superclass*).
- Represents an **"Is-A"** relationship:
  - A `Dog` *is an* `Animal`.
  - A `SavingsAccount` *is a* `BankAccount`.
  - A `Manager` *is an* `Employee`.
- Subclasses can **reuse** existing code, **override** behaviors with custom logic, and **extend** functionality with new attributes and methods.

### 🔍 Hierarchy Introspection Tools
- `isinstance(obj, Class)`: Returns `True` if `obj` is an instance of `Class` or any subclass in its hierarchy.
- `issubclass(SubClass, SuperClass)`: Returns `True` if `SubClass` is derived from `SuperClass`.
- `Class.__bases__`: Returns a tuple of immediate parent classes.


In [ ]:
# Base Class
class Vehicle:
    """Base vehicle class."""
    def __init__(self, make: str, model: str, year: int):
        self.make = make
        self.model = model
        self.year = year

    def start_engine(self) -> str:
        return f"{self.year} {self.make} {self.model} engine started."

# Subclass inheriting from Vehicle
class Car(Vehicle):
    """Car is a Vehicle."""
    pass

# Instantiation & introspection
my_car = Car("Toyota", "Corolla", 2024)

print(my_car.start_engine())
print(f"isinstance(my_car, Car):     {isinstance(my_car, Car)}")
print(f"isinstance(my_car, Vehicle): {isinstance(my_car, Vehicle)}")
print(f"issubclass(Car, Vehicle):    {issubclass(Car, Vehicle)}")
print(f"Car.__bases__:               {Car.__bases__}")


---
## 2. Single Inheritance & Method Overriding

### ⚡ Method Overriding & The `super()` Proxy
- **Method Overriding** occurs when a subclass defines a method with the same name as a method in its parent class.
- When calling `super().method(*args)`, Python dynamically delegates the call to the next class in the Method Resolution Order (MRO).
- `super().__init__(*args)` is standard practice to initialize base class state without hardcoding the base class name.


In [ ]:
class Employee:
    """Base Employee class."""
    def __init__(self, name: str, employee_id: str, base_salary: float):
        self.name = name
        self.employee_id = employee_id
        self.base_salary = base_salary

    def calculate_pay(self) -> float:
        """Calculates monthly compensation."""
        return self.base_salary

    def get_details(self) -> str:
        return f"[{self.employee_id}] {self.name} | Pay: ${self.calculate_pay():,.2f}"

class SalariedManager(Employee):
    """Subclass adding management bonus and direct reports."""
    def __init__(self, name: str, employee_id: str, base_salary: float, bonus: float):
        # Explicitly initialize the parent class via super()
        super().__init__(name, employee_id, base_salary)
        self.bonus = bonus
        self.direct_reports: list[str] = []

    # Overriding calculate_pay to include bonus
    def calculate_pay(self) -> float:
        return self.base_salary + self.bonus

    # Extending with new functionality
    def add_report(self, employee_name: str) -> None:
        self.direct_reports.append(employee_name)

    # Overriding get_details and extending parent string
    def get_details(self) -> str:
        parent_details = super().get_details()
        return f"{parent_details} (Manager - {len(self.direct_reports)} reports)"

emp = Employee("Alice Cooper", "E-101", 5000.0)
mgr = SalariedManager("Bob Vance", "M-202", 8000.0, bonus=2500.0)
mgr.add_report("Alice Cooper")
mgr.add_report("Charlie Brown")

print(emp.get_details())
print(mgr.get_details())


---
## 3. Multilevel & Hierarchical Inheritance

### 🪜 Multilevel Inheritance
A class inherits from a derived class, forming a vertical inheritance chain:
`Device` $ightarrow$ `Computer` $ightarrow$ `Laptop`.

### 🌿 Hierarchical Inheritance
Multiple sibling classes inherit from a single common parent:
`Vehicle` $ightarrow$ `Car`, `Vehicle` $ightarrow$ `Truck`, `Vehicle` $ightarrow$ `Motorcycle`.


In [ ]:
# Level 1: Grandparent
class Device:
    def __init__(self, brand: str, power_watts: float):
        self.brand = brand
        self.power_watts = power_watts

    def power_on(self) -> str:
        return f"Device [{self.brand}] powering on ({self.power_watts}W)..."

# Level 2: Parent (inherits from Device)
class Computer(Device):
    def __init__(self, brand: str, power_watts: float, cpu: str, ram_gb: int):
        super().__init__(brand, power_watts)
        self.cpu = cpu
        self.ram_gb = ram_gb

    def run_benchmark(self) -> str:
        return f"{self.brand} running CPU [{self.cpu}] with {self.ram_gb}GB RAM."

# Level 3: Child (inherits from Computer)
class Laptop(Computer):
    def __init__(self, brand: str, power_watts: float, cpu: str, ram_gb: int, battery_hours: float):
        super().__init__(brand, power_watts, cpu, ram_gb)
        self.battery_hours = battery_hours

    def power_on(self) -> str:
        # Extend parent power_on
        base_msg = super().power_on()
        return f"{base_msg} Battery life: {self.battery_hours}h remaining."

# Instantiating the leaf child
my_laptop = Laptop("Dell", 65.0, "Intel Core i7", 32, 8.5)

print(my_laptop.power_on())
print(my_laptop.run_benchmark())
print(f"isinstance of Device?   {isinstance(my_laptop, Device)}")
print(f"isinstance of Computer? {isinstance(my_laptop, Computer)}")


---
## 4. Multiple Inheritance & The Diamond Problem

### 💎 The Classic Diamond Problem
Multiple inheritance occurs when a class inherits from more than one parent.
The **Diamond Problem** arises when two parent classes inherit from a single grandparent class, and a child inherits from both parents:

```text
       ┌───────────┐
       │     A     │  (Grandparent: defines ping())
       └─────┬─────┘
             │
       ┌─────┴─────┐
       │           │
 ┌─────▼─────┐ ┌───▼───────┐
 │     B     │ │     C     │  (Both override ping())
 └─────┬─────┘ └───┬───────┘
       │           │
       └─────┬─────┘
             │
       ┌─────▼─────┐
       │     D     │  (Inherits from B, C: which ping() runs?)
       └───────────┘
```

In C++, this can lead to duplication or ambiguity. In Python, this is solved completely and deterministically via **C3 Linearization (Method Resolution Order)**.


In [ ]:
class A:
    def ping(self) -> str:
        return "Ping from [A]"

class B(A):
    def ping(self) -> str:
        return "Ping from [B]"

class C(A):
    def ping(self) -> str:
        return "Ping from [C]"

class D(B, C):
    """D inherits from B first, then C."""
    pass

d = D()
print(f"d.ping() result: {d.ping()}")
print(f"D Method Resolution Order:")
for idx, cls in enumerate(D.__mro__):
    print(f"  {idx + 1}. {cls.__name__}")


---
## 5. Method Resolution Order (MRO) & C3 Linearization

### 📐 How Python Computes MRO (C3 Linearization Rules)
Python uses the **C3 Linearization Algorithm** (introduced in Python 2.3) to establish a deterministic linear lookup order for any class hierarchy.

Three fundamental guarantees of C3:
1. **Local Precedence Order**: Children precede their parents; parents are inspected in the exact left-to-right order listed in `class Child(Parent1, Parent2):`.
2. **Monotonicity**: If class $X$ precedes class $Y$ in the MRO of any parent, $X$ will precede $Y$ in the MRO of any derived subclass.
3. **Consistency**: Python rejects illegal/contradictory inheritance trees by raising a `TypeError` at class definition time!


In [ ]:
# Valid complex MRO inspection
class Root: pass
class X(Root): pass
class Y(Root): pass
class Z(Root): pass
class Composite(X, Y, Z): pass

print("Composite MRO:")
print(" -> ".join(cls.__name__ for cls in Composite.__mro__))

# Demonstrating Inconsistent MRO rejection:
# Attempting to define contradictory inheritance order fails immediately!
try:
    class BadA: pass
    class BadB(BadA): pass
    # Contradiction: BadA before BadB, but BadB inherits from BadA!
    class Impossible(BadA, BadB): pass
except TypeError as e:
    print(f"\n[MRO Rejection] Python prevented invalid hierarchy: {e}")


---
## 6. Cooperative Multiple Inheritance with `super()`

### 🤝 The Cooperative Protocol
In multiple inheritance, calling `super().__init__(**kwargs)` does **not** simply call the immediate parent — it calls the **next class in the instance's MRO chain**.

For cooperative multiple inheritance to work cleanly:
1. Every class in the chain must use `super().__init__(**kwargs)`.
2. Every class pops its own expected parameters and passes remaining `**kwargs` upstream.
3. The root class (`object`) accepts empty `kwargs`.


In [ ]:
class BaseCoop:
    def __init__(self, **kwargs):
        # Consumes any leftover kwargs before reaching object.__init__()
        super().__init__()
        print("    [BaseCoop.__init__]")

class AuthPlugin(BaseCoop):
    def __init__(self, auth_token: str, **kwargs):
        super().__init__(**kwargs)
        self.auth_token = auth_token
        print(f"    [AuthPlugin.__init__] token={self.auth_token}")

class CachePlugin(BaseCoop):
    def __init__(self, cache_ttl: int, **kwargs):
        super().__init__(**kwargs)
        self.cache_ttl = cache_ttl
        print(f"    [CachePlugin.__init__] ttl={self.cache_ttl}s")

class ApiClient(AuthPlugin, CachePlugin):
    def __init__(self, endpoint: str, **kwargs):
        super().__init__(**kwargs)
        self.endpoint = endpoint
        print(f"    [ApiClient.__init__] endpoint={self.endpoint}")

print("Initializing ApiClient with cooperative kwargs:")
client = ApiClient(endpoint="https://api.example.com", auth_token="secret_key_123", cache_ttl=300)

print(f"\nClient configuration:")
print(f"Endpoint: {client.endpoint} | Auth: {client.auth_token} | Cache TTL: {client.cache_ttl}")


---
## 7. The Mixin Architectural Pattern

### 🧩 What is a Mixin?
A **Mixin** is a specialized class that provides a bundle of reusable methods to other classes via multiple inheritance.

**Key Characteristics of Mixins**:
- Mixins are **not** meant to be instantiated on their own.
- Mixins usually do **not** define their own `__init__` or store instance state.
- They rely on attributes provided by the host class (`self.id`, `self.name`, `self.__dict__`).
- By convention, Mixin class names end with `Mixin` (e.g., `JSONSerializableMixin`).


In [ ]:
import json
from datetime import datetime

# 1. Reusable JSON Serialization Mixin
class JSONSerializableMixin:
    """Provides robust dictionary and JSON serialization to any class."""
    def to_dict(self) -> dict:
        data = {}
        for key, value in self.__dict__.items():
            if key.startswith("_"):
                continue  # Skip internal attributes
            if isinstance(value, datetime):
                data[key] = value.isoformat()
            else:
                data[key] = value
        return data

    def to_json(self, indent: int = 2) -> str:
        return json.dumps(self.to_dict(), indent=indent)

# 2. Reusable Audit Logging Mixin
class AuditLogMixin:
    """Logs action events with timestamps."""
    def log_action(self, action: str, details: str = "") -> None:
        ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        print(f"[AUDIT {ts}] [{self.__class__.__name__}] {action} -> {details}")

# Host Business Entity combining both Mixins
class UserAccount(JSONSerializableMixin, AuditLogMixin):
    def __init__(self, user_id: str, username: str, email: str):
        self.user_id = user_id
        self.username = username
        self.email = email
        self.created_at = datetime.now()
        self.log_action("CREATED", f"User {self.username} initialized.")

    def change_email(self, new_email: str) -> None:
        old = self.email
        self.email = new_email
        self.log_action("EMAIL_CHANGED", f"From {old} to {new_email}")

user = UserAccount("U-501", "ada_lovelace", "ada@computing.org")
user.change_email("ada.lovelace@alumni.cam.ac.uk")

print("\nSerialized JSON Output:")
print(user.to_json())


---
## 8. Polymorphism: Duck Typing, Abstract Classes, & Protocols

### 🦆 1. Dynamic Polymorphism (Duck Typing)
*"If it walks like a duck and quacks like a duck, it's a duck."*
In Python, functions do not require formal class hierarchies to operate polymorphically — they only require that the passed objects implement the expected methods.

### 📜 2. Formal Polymorphism (`abc.ABC`)
Enforces runtime contract verification and prevents direct instantiation of abstract classes.

### 📐 3. Structural Subtyping (`typing.Protocol`)
Introduced in Python 3.8 (PEP 544), `Protocol` enables **compile-time Duck Typing** with static type checkers (Mypy, Pyright, VS Code).


In [ ]:
from typing import Protocol, runtime_checkable

# Define a Protocol (interface contract)
@runtime_checkable
class Renderable(Protocol):
    def render(self) -> str:
        """Any object with a render() method satisfies this protocol."""
        ...

# Classes that do NOT inherit from a common base class:
class MarkdownDocument:
    def __init__(self, text: str):
        self.text = text

    def render(self) -> str:
        return f"# Markdown Render:\n{self.text}"

class HtmlDocument:
    def __init__(self, content: str):
        self.content = content

    def render(self) -> str:
        return f"<div>{self.content}</div>"

class PlainText:
    def __init__(self, raw: str):
        self.raw = raw

    def render(self) -> str:
        return self.raw

# Polymorphic consumer function
def display_content(document: Renderable) -> None:
    print(f"--- Rendering [{type(document).__name__}] ---")
    print(document.render())

docs: list[Renderable] = [
    MarkdownDocument("## OOP in Python"),
    HtmlDocument("<p>Paragraph</p>"),
    PlainText("Just plain text")
]

for doc in docs:
    print(f"Does {type(doc).__name__} implement Renderable Protocol? {isinstance(doc, Renderable)}")
    display_content(doc)


---
## 9. Composition vs. Inheritance ("Favor Composition")

### ⚖️ The Design Principle
> *"Favor object composition over class inheritance."* — Design Patterns (Gang of Four)

| Dimension | Inheritance ("Is-A") | Composition ("Has-A") |
| :--- | :--- | :--- |
| **Coupling** | **Tight**: Changes to parent can break all subclasses | **Loose**: Components can be swapped independently at runtime |
| **Flexibility** | Static (fixed at compile/definition time) | Dynamic (components can change dynamically) |
| **Encapsulation** | Can expose internal parent implementation details | Strong encapsulation; interacts only via clean component interfaces |
| **Best Used When** | True taxonomic relationship & polymorphic substitution | Building complex objects from swappable feature modules |


In [ ]:
# Component 1: Engine Strategy
class V8GasEngine:
    def start(self) -> str:
        return "V8 Gas Engine roaring to life (Combustion)."

class ElectricMotor:
    def start(self) -> str:
        return "Electric Motor humming silently (0 emissions)."

# Component 2: GPS Strategy
class StandardGPS:
    def navigate(self, destination: str) -> str:
        return f"Standard GPS calculating route to '{destination}'."

class AutonomousNavigation:
    def navigate(self, destination: str) -> str:
        return f"Autonomous AI LiDAR & Camera pathfinding to '{destination}'."

# Car uses COMPOSITION (Has-A Engine, Has-A Navigation)
class VehicleComposite:
    def __init__(self, model: str, engine, navigation):
        self.model = model
        self.engine = engine          # Composition: Car HAS-AN engine
        self.navigation = navigation  # Composition: Car HAS-A navigation system

    def drive_to(self, destination: str) -> None:
        print(f"[{self.model}]")
        print(f"  Engine: {self.engine.start()}")
        print(f"  Nav:    {self.navigation.navigate(destination)}")

# Easily assemble different configurations without class explosion:
muscle_car = VehicleComposite("Mustang GT", V8GasEngine(), StandardGPS())
ev_car = VehicleComposite("Tesla Model S", ElectricMotor(), AutonomousNavigation())

muscle_car.drive_to("Detroit, MI")
print()
ev_car.drive_to("Silicon Valley, CA")


---
## 10. Real-World Case Studies & Architectural Patterns

---

### 10.1 Enterprise Multi-Channel Notifier Hierarchy
Base class with template method pattern, specific channel subclasses, and a retry Mixin.


In [ ]:
from abc import ABC, abstractmethod
import time

class RetryMixin:
    """Provides automatic retry capabilities for network operations."""
    def execute_with_retry(self, operation, max_attempts: int = 3):
        for attempt in range(1, max_attempts + 1):
            try:
                return operation()
            except Exception as e:
                if attempt == max_attempts:
                    print(f"    [RETRY FAILED] All {max_attempts} attempts failed: {e}")
                    raise
                print(f"    [RETRY {attempt}/{max_attempts}] Operation failed, retrying...")
                time.sleep(0.01)

class BaseNotifier(ABC):
    def __init__(self, channel_name: str):
        self.channel_name = channel_name

    @abstractmethod
    def format_payload(self, recipient: str, message: str) -> dict:
        pass

    @abstractmethod
    def deliver(self, payload: dict) -> bool:
        pass

    def send_notification(self, recipient: str, message: str) -> bool:
        """Template Method defining the standardized notification workflow."""
        payload = self.format_payload(recipient, message)
        print(f"[{self.channel_name}] Prepared payload for {recipient}")
        return self.deliver(payload)

class EmailNotifier(BaseNotifier, RetryMixin):
    def __init__(self, smtp_server: str):
        super().__init__("Email-Channel")
        self.smtp_server = smtp_server

    def format_payload(self, recipient: str, message: str) -> dict:
        return {"to": recipient, "subject": "System Alert", "body": message}

    def deliver(self, payload: dict) -> bool:
        def _send():
            print(f"  --> Delivered SMTP message to {payload['to']} via {self.smtp_server}")
            return True
        return self.execute_with_retry(_send)

class SlackNotifier(BaseNotifier, RetryMixin):
    def __init__(self, webhook_url: str):
        super().__init__("Slack-Channel")
        self.webhook_url = webhook_url

    def format_payload(self, recipient: str, message: str) -> dict:
        return {"channel": recipient, "text": f":bell: {message}"}

    def deliver(self, payload: dict) -> bool:
        def _send():
            print(f"  --> Posted webhook to Slack channel {payload['channel']}")
            return True
        return self.execute_with_retry(_send)

# Polymorphic notification dispatch
channels: list[BaseNotifier] = [
    EmailNotifier("smtp.corporate.net"),
    SlackNotifier("https://hooks.slack.com/services/T00/B00/X00")
]

for ch in channels:
    ch.send_notification("dev-ops-team", "Production deployment successful.")


---
### 10.2 Custom Container with `collections.abc`
Inheriting from `collections.abc.MutableSequence` to build a production-grade, type-safe list.


In [ ]:
from collections.abc import MutableSequence

class TypedList(MutableSequence):
    """A strictly typed list that only permits elements of an approved type."""

    def __init__(self, allowed_type: type, initial_elements = None):
        self._allowed_type = allowed_type
        self._inner_list: list = []
        if initial_elements:
            for item in initial_elements:
                self.append(item)

    def _validate(self, value):
        if not isinstance(value, self._allowed_type):
            raise TypeError(f"Invalid element type: expected {self._allowed_type.__name__}, got {type(value).__name__}")

    # Required MutableSequence abstract methods:
    def __getitem__(self, index):
        return self._inner_list[index]

    def __setitem__(self, index, value):
        self._validate(value)
        self._inner_list[index] = value

    def __delitem__(self, index):
        del self._inner_list[index]

    def __len__(self) -> int:
        return len(self._inner_list)

    def insert(self, index: int, value) -> None:
        self._validate(value)
        self._inner_list.insert(index, value)

    def __repr__(self) -> str:
        return f"TypedList({self._allowed_type.__name__}, {self._inner_list})"

# Demo TypedList
int_list = TypedList(int, [10, 20, 30])
int_list.append(40)
int_list.extend([50, 60])

print(f"int_list: {int_list}")
print(f"Sum: {sum(int_list)} | Length: {len(int_list)}")

# Attempting to insert invalid type
try:
    int_list.append("invalid_string")
except TypeError as e:
    print(f"[Type Enforcement] Caught error: {e}")


---
## 11. Common Pitfalls & Anti-Patterns

### ❌ Pitfall 1: Hardcoding Parent Initializers Instead of `super()`
Calling `Parent.__init__(self)` explicitly breaks the cooperative MRO chain and can result in classes being initialized multiple times or skipped entirely.

### ❌ Pitfall 2: Violating the Liskov Substitution Principle (LSP)
If class `B` is a subclass of `A`, any code expecting an object of class `A` should work correctly with an object of class `B` without needing special checks. 
*Anti-pattern: Subclass throwing exceptions for base class methods or expecting completely different parameter types.*

### ❌ Pitfall 3: The "Yo-Yo" Problem (Excessively Deep Hierarchies)
When inheritance hierarchies are 5+ levels deep, understanding the flow of control requires jumping up and down across files like a yo-yo. Keep hierarchies shallow ($\le 3$ levels) and use composition for additional features.


---
## 12. Hands-On Interactive Challenges

---

### 🎯 Challenge 1: Geometric Shape Hierarchy
Create an abstract base class `Shape` (`abc.ABC`) with:
- Abstract methods: `area() -> float` and `perimeter() -> float`.
- Concrete subclasses: `Rectangle(width, height)` and `Circle(radius)`.
- Automated test assertions verify polymorphic calculation.


In [ ]:
from abc import ABC, abstractmethod
import math

class Shape(ABC):
    @abstractmethod
    def area(self) -> float:
        pass

    @abstractmethod
    def perimeter(self) -> float:
        pass

class Rectangle(Shape):
    def __init__(self, width: float, height: float):
        if width <= 0 or height <= 0:
            raise ValueError("Dimensions must be positive.")
        self.width = float(width)
        self.height = float(height)

    def area(self) -> float:
        return self.width * self.height

    def perimeter(self) -> float:
        return 2 * (self.width + self.height)

class Circle(Shape):
    def __init__(self, radius: float):
        if radius <= 0:
            raise ValueError("Radius must be positive.")
        self.radius = float(radius)

    def area(self) -> float:
        return math.pi * (self.radius ** 2)

    def perimeter(self) -> float:
        return 2 * math.pi * self.radius

# Automated verification test
rect = Rectangle(4, 5)
circ = Circle(3)

assert rect.area() == 20.0
assert rect.perimeter() == 18.0
assert round(circ.area(), 2) == 28.27
assert round(circ.perimeter(), 2) == 18.85

print("[OK] Challenge 1 Passed!")


---
### 🎯 Challenge 2: Logger Hierarchy with Timestamp Mixin
Implement:
- `TimestampMixin` with `format_message(msg: str) -> str` prefixing `[YYYY-MM-DD HH:MM:SS]`.
- Abstract `BaseLogger(ABC)` with `log(msg: str)`.
- Concrete `MemoryLogger(BaseLogger, TimestampMixin)` that stores formatted messages in a list `self.logs`.


In [ ]:
from abc import ABC, abstractmethod
from datetime import datetime

class TimestampMixin:
    def format_message(self, msg: str) -> str:
        ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        return f"[{ts}] {msg}"

class BaseLogger(ABC):
    @abstractmethod
    def log(self, message: str) -> None:
        pass

class MemoryLogger(BaseLogger, TimestampMixin):
    def __init__(self):
        self.logs: list[str] = []

    def log(self, message: str) -> None:
        formatted = self.format_message(message)
        self.logs.append(formatted)

# Automated verification test
logger = MemoryLogger()
logger.log("Service started")
logger.log("Database connection established")

assert len(logger.logs) == 2
assert "Service started" in logger.logs[0]
assert "Database connection established" in logger.logs[1]

print("[OK] Challenge 2 Passed! Sample log:", logger.logs[0])


---
### 🎯 Challenge 3: Cooperative Bank Account Hierarchy
Implement:
- Base class `BankAccount(balance)` with `withdraw(amount)`.
- `OverdraftProtectionMixin` that allows withdrawals exceeding balance up to `overdraft_limit`.
- Derived class `PremiumChecking(OverdraftProtectionMixin, BankAccount)`.


In [ ]:
class BankAccount:
    def __init__(self, balance: float, **kwargs):
        super().__init__(**kwargs)
        self.balance = float(balance)

    def withdraw(self, amount: float) -> bool:
        if amount <= self.balance:
            self.balance -= amount
            return True
        return False

class OverdraftProtectionMixin:
    def __init__(self, overdraft_limit: float = 500.0, **kwargs):
        super().__init__(**kwargs)
        self.overdraft_limit = float(overdraft_limit)

    def withdraw(self, amount: float) -> bool:
        # Check standard withdrawal first
        if super().withdraw(amount):
            return True
        # Check overdraft capability
        if amount <= (self.balance + self.overdraft_limit):
            self.balance -= amount
            return True
        return False

class PremiumChecking(OverdraftProtectionMixin, BankAccount):
    def __init__(self, balance: float, overdraft_limit: float = 500.0):
        super().__init__(balance=balance, overdraft_limit=overdraft_limit)

# Automated verification test
acc = PremiumChecking(balance=100.0, overdraft_limit=300.0)

# Withdraw within balance
assert acc.withdraw(50.0) is True
assert acc.balance == 50.0

# Withdraw using overdraft limit (50 balance + 300 limit = 350 max)
assert acc.withdraw(200.0) is True
assert acc.balance == -150.0

# Exceeding overdraft limit fails
assert acc.withdraw(250.0) is False
assert acc.balance == -150.0

print("[OK] Challenge 3 Passed! Final balance:", acc.balance)


---
## 13. Quick Reference Card & Summary

### 💡 Complete Inheritance & Polymorphism Reference Cell


In [ ]:
# ============================================================
# PYTHON INHERITANCE & POLYMORPHISM — QUICK REFERENCE
# ============================================================

from abc import ABC, abstractmethod
from typing import Protocol, runtime_checkable

# 1. Abstract Base Class
class BaseService(ABC):
    def __init__(self, service_name: str):
        self.service_name = service_name

    @abstractmethod
    def execute(self) -> str:
        """Subclasses MUST implement."""
        pass

# 2. Mixin Class
class LoggingMixin:
    def log(self, message: str) -> None:
        print(f"[{self.__class__.__name__}] {message}")

# 3. Subclass with Single Inheritance & Mixin
class PaymentService(BaseService, LoggingMixin):
    def __init__(self, service_name: str, gateway: str):
        super().__init__(service_name)
        self.gateway = gateway

    def execute(self) -> str:
        self.log(f"Processing transaction via {self.gateway}")
        return "SUCCESS"

# 4. Protocol for Duck Typing
@runtime_checkable
class Executable(Protocol):
    def execute(self) -> str: ...

# Instantiation & Validation
service = PaymentService("CheckoutService", "Stripe")
result = service.execute()

print(f"MRO: {[c.__name__ for c in PaymentService.__mro__]}")
print(f"Is Executable Protocol? {isinstance(service, Executable)}")
print(f"Is BaseService subclass? {issubclass(PaymentService, BaseService)}")


### 📊 Master Inheritance Cheat Sheet

| Concept | Syntax / Mechanism | Key Use Case & Purpose |
| :--- | :--- | :--- |
| **Single Inheritance** | `class Sub(Base):` | Extending or specializing a base class |
| **Method Overriding** | `def method(self): super().method()` | Customizing behavior while preserving parent logic |
| **Multiple Inheritance** | `class Sub(ParentA, ParentB):` | Combining capabilities from multiple parent sources |
| **MRO Inspection** | `Class.__mro__` / `Class.mro()` | Examining the C3 Linearization lookup order |
| **Cooperative super()** | `super().__init__(**kwargs)` | Ensuring all parents in MRO are cleanly initialized |
| **Mixin Pattern** | `class Mixin:` (no state) | Modular code reuse across unrelated classes |
| **Abstract Base Class** | `class Base(ABC): @abstractmethod` | Enforcing strict API implementation contracts |
| **Protocols** | `class Proto(Protocol):` | Static type checking and structural Duck Typing |
| **Composition** | `self.component = Component()` | Loosely coupled, dynamic runtime modularity |

---
*Next up in OOP Series → **Magic (Dunder) Methods (`dunder_methods.ipynb`)***
